# Fuzzy Matching addresses to get matching records from HPD and NYCHA violation data

## Making the NYCHA address column

In [3]:
import pandas as pd
from joblib import dump

In [8]:
nycha_df = pd.read_csv("../data/nycha.csv")
hpd_df = pd.read_parquet("../data/hpd_violations.parquet")

### Address format:

Primary House Number, Primary Street Name, Primary Borough Name, Primary Postcode (No records have null or blank values for these columns)

Coordinates will be used for extra spatial confirmation later:

Longitude, Latitude

In [9]:
nycha_df['Full_Address'] = (
    nycha_df['Primary House Number'].astype(str) + " " +
    nycha_df['Primary Street Name'].astype(str) + " " + 
    nycha_df['Primary Borough Name'].astype(str) + " " +
    nycha_df['Primary Postcode'].astype(str)
)

In [ ]:
nycha_df.tail()

,Violation ID,Primary Building Identifier,Primary Boro Identifier,Primary Borough Name,Primary House Number,Primary Low House Number,Primary High House Number,Primary Street Name,Primary Postcode,Development Name,...,Issued in Error,Latitude,Longitude,Community Board,Council District,BIN,BBL,Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020),Full_Address
2386,18691906,807796,3,BROOKLYN,422,422,430,BLAKE AVENUE,11212,VAN DYKE I,...,N,40.666473,-73.903929,316,41,3328040.0,3.037770e+09,910,BK1602,422 BLAKE AVENUE BROOKLYN 11212
2387,18691907,807796,3,BROOKLYN,422,422,430,BLAKE AVENUE,11212,VAN DYKE I,...,N,40.666473,-73.903929,316,41,3328040.0,3.037770e+09,910,BK1602,422 BLAKE AVENUE BROOKLYN 11212
2388,18692344,989840,1,MANHATTAN,505,505,513,EAST 120 STREET,10035,WAGNER,...,N,40.797582,-73.931095,111,8,1081286.0,1.018080e+09,192,MN1102,505 EAST 120 STREET MANHATTAN 10035
2389,18701333,807796,3,BROOKLYN,422,422,430,BLAKE AVENUE,11212,VAN DYKE I,...,N,40.666473,-73.903929,316,41,3328040.0,3.037770e+09,910,BK1602,422 BLAKE AVENUE BROOKLYN 11212
2390,18701334,807796,3,BROOKLYN,422,422,430,BLAKE AVENUE,11212,VAN DYKE I,...,N,40.666473,-73.903929,316,41,3328040.0,3.037770e+09,910,BK1602,422 BLAKE AVENUE BROOKLYN 11212


### Address columns for HPD to get the same format as above

HouseNumber StreetName Borough Postcode

Coordinates will be used for extra spatial confirmation later:

Longitude, Latitude


### Before creating full_address column for HPD here are the resultsof data integrity checks for these columns ^

#### Nulls?

7524 null Postcodes

The remaining columns have valid values 

In [10]:
# Fill the null postal codes  so they don't break concatenation
hpd_df['Postcode'] = hpd_df['Postcode'].fillna("").astype(str).str.strip()

In [11]:
hpd_df['Full_Address'] = (
    hpd_df['HouseNumber'].astype(str) + " " +
    hpd_df['StreetName'].astype(str) + " " + 
    hpd_df['Borough'].astype(str) + " " +
    hpd_df['Postcode'].astype(str)
)

In [12]:
# remove the trailing .0 at the float -> str in postcode
hpd_df['Full_Address'] = hpd_df['Full_Address'].str.replace(r'\.0$', '', regex=True)
hpd_df['Borough'].unique()

array(['BROOKLYN', 'MANHATTAN', 'BRONX', 'QUEENS', 'STATEN ISLAND'],
      dtype=object)

### Create a new DF where each record is a address fuzzy matched across both data sets
Includes column for NYCHA violation IDs and HPD violation IDs

Since the HPD parquet is so big - split them into DFs by borough first

In [13]:
from rapidfuzz import fuzz, process

In [ ]:
# Only fuzzy match within the same house number


In [14]:
hpd_brooklyn = hpd_df[hpd_df['Borough'] == "BROOKLYN"]
hpd_manhattan = hpd_df[hpd_df['Borough'] == "MANHATTAN"]
hpd_queens = hpd_df[hpd_df['Borough'] == "QUEENS"]
hpd_bronx = hpd_df[hpd_df['Borough'] == "BRONX"]
hpd_si = hpd_df[hpd_df['Borough'] == "STATEN ISLAND"]



In [ ]:
# borough_map_addresses = {
#     "BROOKLYN": hpd_brooklyn['Full_Address'],
#     "BRONX": hpd_bronx['Full_Address'],
#     "MANHATTAN": hpd_manhattan['Full_Address'],
#     "QUEENS": hpd_queens['Full_Address'],
#     "STATEN ISLAND": hpd_si['Full_Address']
# }

In [15]:
borough_map = {
    "BROOKLYN": hpd_brooklyn,
    "BRONX": hpd_bronx,
    "MANHATTAN": hpd_manhattan,
    "QUEENS": hpd_queens,
    "STATEN ISLAND": hpd_si
}

In [16]:
dump(borough_map, 'borough_map.joblib')

['borough_map.joblib']

In [ ]:
borough_map["BROOKLYN"]

,ViolationID,BuildingID,RegistrationID,BoroID,Borough,HouseNumber,LowHouseNumber,HighHouseNumber,StreetName,StreetCode,...,Latitude,Longitude,CommunityBoard,CouncilDistrict,CensusTract,BIN,BBL,NTA,BIN_key,Full_Address
10746935,6041079,380474,360074,3,BROOKLYN,405,405,405,SUYDAM STREET,83930,...,40.705675,-73.920229,4.0,34.0,445.0,3073015.0,3.032110e+09,Bushwick (West),3073015,405 SUYDAM STREET BROOKLYN 11237
10746969,7080427,1012097,391037,3,BROOKLYN,5,5,25,EAST 93 STREET,37930,...,40.662609,-73.927104,17.0,41.0,882.0,3346863.0,3.045950e+09,East Flatbush-Remsen Village,3346863,5 EAST 93 STREET BROOKLYN 11212
10746970,7478024,339272,324596,3,BROOKLYN,80,74,80,MONROE STREET,61730,...,40.684851,-73.955889,3.0,36.0,229.0,3057121.0,3.019880e+09,Bedford-Stuyvesant (West),3057121,80 MONROE STREET BROOKLYN 11216
10746982,8546834,340160,333949,3,BROOKLYN,241,241,241,MONTROSE AVENUE,62430,...,40.707729,-73.939141,1.0,34.0,49302.0,3071182.0,3.030550e+09,East Williamsburg,3071182,241 MONTROSE AVENUE BROOKLYN 11206
10746983,8546836,340160,333949,3,BROOKLYN,241,241,241,MONTROSE AVENUE,62430,...,40.707729,-73.939141,1.0,34.0,49302.0,3071182.0,3.030550e+09,East Williamsburg,3071182,241 MONTROSE AVENUE BROOKLYN 11206


In [ ]:
# For each NYCHA address, find best HPD match

results = []

def calc_fuzzy_match(query, hpd_addresses):
    match, score, hpd_index = process.extractOne(query, hpd_addresses)
    return match, score, hpd_index


for idx, row in nycha_df.iterrows():
    borough_name = row['Primary Borough Name']
    hpd_borough_df = borough_map.get(borough_name)

    if hpd_borough_df is None:
        continue

    nycha_house_num = str(row['Primary House Number'])
    filtered = hpd_borough_df[hpd_borough_df['HouseNumber'].astype(str) == nycha_house_num]
    if filtered.empty:
        filtered = hpd_borough_df

    hpd_addresses = filtered['Full_Address'].tolist()


    match, score, hpd_index = calc_fuzzy_match(row['Full_Address'], hpd_addresses)
    results.append({
        'nycha_idx': idx,
        'hpd_inx': hpd_index,
        'nycha_address': row['Full_Address'],
        'hpd_address': match,
        'score': score,
        'used_fallback': filtered.empty
        

    })


In [ ]:
results_df = pd.DataFrame(results)

total = len(results_df)
high_confidence = results_df[results_df['score'] >= 99]
medium_confidence = results_df[(results_df['score'] >= 70) & (results_df['score'] < 90)]
low_confidence = results_df[results_df['score'] < 70]

fallback = results_df[results_df['used_fallback'] == True]
filtered = results_df[results_df['used_fallback'] == False]

print(f"Total matched:       {total}")
print(f"High (>=90):         {len(high_confidence)} ({len(high_confidence)/total*100:.1f}%)")
print(f"Medium (70-89):      {len(medium_confidence)} ({len(medium_confidence)/total*100:.1f}%)")
print(f"Low (<70):           {len(low_confidence)} ({len(low_confidence)/total*100:.1f}%)")
print(f"---")
print(f"House number filter: {len(filtered)} ({len(filtered)/total*100:.1f}%)")
print(f"Fallback:            {len(fallback)} ({len(fallback)/total*100:.1f}%)")

Total matched:       2391
High (>=90):         785 (32.8%)
Medium (70-89):      893 (37.3%)
Low (<70):           103 (4.3%)
---
House number filter: 2391 (100.0%)
Fallback:            0 (0.0%)


In [ ]:
high_confidence.shape

(785, 6)

In [ ]:

#medium confidence ones do not look good so here we're just saving high confidence ones
high_confidence.to_csv('initial_address_matches.csv', index=False)

In [ ]:
high_confidence.sort_values(by='score', ascending=True)

,nycha_idx,hpd_inx,nycha_address,hpd_address,score,used_fallback
0,0,4674,1 LORRAINE STREET BROOKLYN 11231,1 LORRAINE STREET BROOKLYN 11231,100.0,False
1523,1523,1067,218 EAST 115 STREET MANHATTAN 10029,218 EAST 115 STREET MANHATTAN 10029,100.0,False
1524,1524,1,2703 WEST 33 STREET BROOKLYN 11224,2703 WEST 33 STREET BROOKLYN 11224,100.0,False
1525,1525,833,2201 SURF AVENUE BROOKLYN 11224,2201 SURF AVENUE BROOKLYN 11224,100.0,False
1526,1526,833,2201 SURF AVENUE BROOKLYN 11224,2201 SURF AVENUE BROOKLYN 11224,100.0,False
...,...,...,...,...,...,...
764,764,72,116-30 GUY R BREWER BOULEVARD QUEENS 11434,116-30 GUY R BREWER BOULEVARD QUEENS 11434,100.0,False
765,765,72,116-30 GUY R BREWER BOULEVARD QUEENS 11434,116-30 GUY R BREWER BOULEVARD QUEENS 11434,100.0,False
766,766,72,116-30 GUY R BREWER BOULEVARD QUEENS 11434,116-30 GUY R BREWER BOULEVARD QUEENS 11434,100.0,False
768,768,3930,35 DWIGHT STREET BROOKLYN 11231,35 DWIGHT STREET BROOKLYN 11231,100.0,False


In [ ]:
high_confidence = high_confidence.drop_duplicates(subset=['nycha_address', 'hpd_address'])

A cleaned high_confidence DF with no duplicates has 217 records. This is only 10 addresses less than the original nycha_df number of unique addresses.
Only 10 addresses will not enjoy historic data from the HPD dataset

In [ ]:
high_confidence.shape

(73, 6)

In [ ]:
len(nycha_df['Full_Address'].unique())

227

In [ ]:
high_confidence.to_csv("cleaned_fuzzy_matching.csv", index=False)

In [ ]:
high_confidence.head()

,nycha_idx,hpd_inx,nycha_address,hpd_address,score
0,0,3195153,1 LORRAINE STREET BROOKLYN 11231,1 LORRAINE STREET BROOKLYN 11231,100.000000
3,3,778118,621 EAST 105 STREET BROOKLYN 11236,621 EAST 102 STREET BROOKLYN 11236,97.058824
10,10,14153,1605 EAST 174 STREET BRONX 10472,1670 EAST 174 STREET BRONX 10472,96.875000
12,12,65564,147 RICHARDS STREET BROOKLYN 11231,179 RICHARDS STREET BROOKLYN 11231,97.058824
16,16,205173,1591 PARK AVENUE MANHATTAN 10029,1351 PARK AVENUE MANHATTAN 10029,96.875000


## Prepping violation and address dataframe for export to adding_vios notebook

1. Create dictionary of violations with address by borough
2. Keep only fuzzy matched addresses from ^ dict

In [17]:
hpd_vios_by_borough = {
    "BROOKLYN": hpd_brooklyn[['ViolationID','Full_Address']],
    "BRONX": hpd_bronx[['ViolationID','Full_Address']],
    "MANHATTAN": hpd_manhattan[['ViolationID','Full_Address']],
    "QUEENS": hpd_queens[['ViolationID','Full_Address']],
    "STATEN ISLAND": hpd_si[['ViolationID','Full_Address']]
}



In [18]:
# same as high_confidence var 
matched_addresses = pd.read_csv("../processed_data/cleaned_fuzzy_matching.csv")
matched_addresses.tail()

,nycha_idx,hpd_inx,nycha_address,hpd_address,score,used_fallback
68,2126,0,2692 FREDERICK DOUGLASS BOULEVARD MANHATTAN 10030,2692 FREDERICK DOUGLASS BOULEVARD MANHATTAN 10030,100.0,False
69,2128,48,1299 AMSTERDAM AVENUE MANHATTAN 10027,1299 AMSTERDAM AVENUE MANHATTAN 10027,100.0,False
70,2184,3309,85 LORRAINE STREET BROOKLYN 11231,85 LORRAINE STREET BROOKLYN 11231,100.0,False
71,2337,92,430 EAST 105 STREET MANHATTAN 10029,430 EAST 105 STREET MANHATTAN 10029,100.0,False
72,2365,1518,230 WEST 129 STREET MANHATTAN 10027,230 WEST 129 STREET MANHATTAN 10027,100.0,False


In [27]:
results = {}
for borough, df in hpd_vios_by_borough.items():
    # for every matched address add all violationIds where the addresses match into one column 
    merged_vios_with_adds = pd.merge(
        df,
        matched_addresses,
        left_on='Full_Address',
        right_on='hpd_address',
        how='inner'
    )

    # group ids based on address 
    grouped_vios_by_add = merged_vios_with_adds.groupby('hpd_address')['ViolationID'].apply(list).reset_index()

    results[borough] = grouped_vios_by_add



In [20]:
for boroughs, df in results.items():
    results[boroughs] = df.rename(columns={'ViolationID': 'HPD_ViolationID'})   

In [21]:
results['BROOKLYN']

,hpd_address,HPD_ViolationID
0,1 LORRAINE STREET BROOKLYN 11231,"[5655038, 5655039, 5655040, 5655041, 8119910, ..."
1,10602 GLENWOOD ROAD BROOKLYN 11236,"[9547154, 9547246]"
2,123 LORRAINE STREET BROOKLYN 11231,"[1913653, 1913654, 1913655, 1913656]"
3,161 BUSH STREET BROOKLYN 11231,[11683967]
4,177 SANDS STREET BROOKLYN 11201,"[6329476, 6329479, 6329481, 6329485, 6329489, ..."
5,200 THROOP AVENUE BROOKLYN 11206,"[10243168, 10243169, 10384461]"
6,220 THROOP AVENUE BROOKLYN 11206,"[10019755, 10019757, 10019759]"
7,2201 SURF AVENUE BROOKLYN 11224,"[9480187, 9480196, 9480200]"
8,234 SANDS STREET BROOKLYN 11201,[8341379]
9,2703 WEST 33 STREET BROOKLYN 11224,"[11100618, 11099883, 11099884, 11100617]"


In [62]:
nycha_df.head()

,Violation ID,Primary Building Identifier,Primary Boro Identifier,Primary Borough Name,Primary House Number,Primary Low House Number,Primary High House Number,Primary Street Name,Primary Postcode,Development Name,...,Issued in Error,Latitude,Longitude,Community Board,Council District,BIN,BBL,Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020),Full_Address
0,18220152,808768,3,BROOKLYN,1,1,17,LORRAINE STREET,11231,RED HOOK WEST,...,N,40.675118,-74.009568,306,38,3326207.0,3.005330e+09,85,BK0601,1 LORRAINE STREET BROOKLYN 11231
1,18220153,808768,3,BROOKLYN,1,1,17,LORRAINE STREET,11231,RED HOOK WEST,...,N,40.675118,-74.009568,306,38,3326207.0,3.005330e+09,85,BK0601,1 LORRAINE STREET BROOKLYN 11231
2,18220154,808768,3,BROOKLYN,1,1,17,LORRAINE STREET,11231,RED HOOK WEST,...,N,40.675118,-74.009568,306,38,3326207.0,3.005330e+09,85,BK0601,1 LORRAINE STREET BROOKLYN 11231
3,18223815,286990,3,BROOKLYN,621,615,621,EAST 105 STREET,11236,BREUKELEN,...,N,40.650186,-73.897621,318,42,3321415.0,3.081740e+09,982,BK1803,621 EAST 105 STREET BROOKLYN 11236
4,18223816,286990,3,BROOKLYN,621,615,621,EAST 105 STREET,11236,BREUKELEN,...,N,40.650186,-73.897621,318,42,3321415.0,3.081740e+09,982,BK1803,621 EAST 105 STREET BROOKLYN 11236


In [48]:
#repeat for NYCHA df
merged_vios_with_adds = pd.merge(
    nycha_df,
    matched_addresses,
    left_on='Full_Address',
    right_on='nycha_address',
    how='left'
)

group_nycha_vios_by_add = merged_vios_with_adds.groupby('Full_Address')['Violation ID'].apply(list).reset_index()
group_nycha_vios_by_add = group_nycha_vios_by_add.rename(columns={'Violation ID': 'NYCHA_ViolationID'})


In [49]:
group_nycha_vios_by_add

,Full_Address,NYCHA_ViolationID
0,1 LORRAINE STREET BROOKLYN 11231,"[18220152, 18220153, 18220154, 18220149, 18220..."
1,1-25 ASTORIA BOULEVARD QUEENS 11102,"[18376491, 18376493, 18376494, 18376495, 18376..."
2,1008 ST MARKS AVENUE BROOKLYN 11213,"[18327321, 18327307, 18327308, 18327309, 18327..."
3,106-24 159 STREET QUEENS 11433,"[18517390, 18517398, 18517399, 18517401, 18517..."
4,106-35 159 STREET QUEENS 11433,"[18635309, 18635310, 18635311, 18635312, 18635..."
...,...,...
222,90 AVENUE D MANHATTAN 10009,"[18515963, 18545433, 18545434, 18545446, 18545..."
223,90 PITT STREET MANHATTAN 10002,"[18280020, 18280021, 18280022, 18280023, 18286..."
224,915 WARING AVENUE BRONX 10469,"[18636645, 18636642, 18636643, 18636644]"
225,920 EAST 6 STREET MANHATTAN 10009,"[18393660, 18393661, 18393662, 18393663, 18393..."


In [24]:
#SPLITTING NYCHA VIOS BY BOROUGH TO MERGE W HPD VIOS
nycha_brooklyn = nycha_df[nycha_df['Primary Borough Name'] == "BROOKLYN"]
nycha_manhattan = nycha_df[nycha_df['Primary Borough Name'] == "MANHATTAN"]
nycha_queens = nycha_df[nycha_df['Primary Borough Name'] == "QUEENS"]
nycha_bronx = nycha_df[nycha_df['Primary Borough Name'] == "BRONX"]
nycha_si = nycha_df[nycha_df['Primary Borough Name'] == "STATEN ISLAND"]



In [ ]:
nyhca_borough_map = {
    "BROOKLYN": nycha_brooklyn,
    "BRONX": nycha_bronx,
    "MANHATTAN": nycha_manhattan,
    "QUEENS": nycha_queens,
    "STATEN ISLAND": nycha_si
}
len(nyhca_borough_map['']['Full_Address'].unique())

102

In [50]:
final_results = {}

for borough, hpd_df in results.items():
    # 1. Get the NYCHA records for this borough
    nycha_borough = group_nycha_vios_by_add[
        group_nycha_vios_by_add['Full_Address'].str.contains(borough, case=False)
    ].copy()
    
    # 2. Use LEFT join starting with NYCHA
    # This ensures NYCHA stays, and HPD is the "supplement"
    merged = pd.merge(
        nycha_borough,
        hpd_df,
        left_on='Full_Address',
        right_on='hpd_address',
        how='left'
    ).drop(columns=['hpd_address'])
    
    final_results[borough] = merged

In [68]:
for borough, df in final_results.items():
    df.rename(columns={'HPD_ViolationID': 'HPD_ViolationIDs'},inplace=True)
    df.rename(columns={'NYCHA_ViolationID': 'NYCHA_ViolationIDs'},inplace=True)


In [69]:
final_results['QUEENS']

,Full_Address,NYCHA_ViolationIDs,HPD_ViolationIDs
0,1-25 ASTORIA BOULEVARD QUEENS 11102,"[18376491, 18376493, 18376494, 18376495, 18376...",NaN
1,106-24 159 STREET QUEENS 11433,"[18517390, 18517398, 18517399, 18517401, 18517...",NaN
2,106-35 159 STREET QUEENS 11433,"[18635309, 18635310, 18635311, 18635312, 18635...",NaN
3,116-30 GUY R BREWER BOULEVARD QUEENS 11434,"[18365449, 18365450, 18365451, 18365453, 18365...",[3980277]
4,116-40 GUY R BREWER BOULEVARD QUEENS 11434,"[18510187, 18510188, 18496095, 18496096, 18496...",NaN
5,133-16 ROOSEVELT AVENUE QUEENS 11354,"[18658551, 18658552]","[343808, 343809, 343810]"
6,133-36 ROOSEVELT AVENUE QUEENS 11354,"[18495727, 18495732, 18495728, 18495729, 18495...",NaN
7,14-68 BEACH CHANNEL DRIVE QUEENS 11691,"[18569184, 18569182, 18569180, 18569181, 18569...",NaN
8,154-02 JEWEL AVENUE QUEENS 11367,"[18298789, 18298790, 18298791, 18298792, 18298...","[6379872, 6379873, 6379868, 6379869, 6379870, ..."
9,154-05 71 AVENUE QUEENS 11367,"[18502297, 18502299, 18502300, 18502301, 18502...",NaN


In [70]:
import json
# Convert each DF to a dict, then save the whole thing
final_json = {borough: df.to_dict(orient='records') for borough, df in results.items()}

with open('data.json', 'w') as f:
    json.dump(final_json, f)

In [71]:
## Creating drilled down dfs -> json 

# Get every unique address that actually has a matched violation
matched_address_list = []
for df in results.values():
    matched_address_list.extend(df['hpd_address'].unique())

matched_address_set = set(matched_address_list) # Using a set makes lookup O(1) speed

In [75]:
len(matched_address_set)

73

In [ ]:
subset_list = []

for borough, df in borough_map.items():
    # Only grab the rows where the address was in our 'matched' list
    found_rows = df[df['Full_Address'].isin(matched_address_set)]
    subset_list.append(found_rows)

# Combine them into one "Master Lookup" for the web app
hpd_lookup_final = pd.concat(subset_list, ignore_index=True)


In [ ]:
master_data = {
    "hpd_vios": hpd_lookup_final.to_dict(orient='records'),
    "nycha_vios": nycha_df.to_dict(orient='records')
}

with open('data.json', 'w') as f:
    json.dump(master_data, f)